In [3]:
import json
from pathlib import Path

import torch
from tqdm.auto import tqdm

from load_dataset import (
    augment_torch_batch,
    build_dataset,
    format_torch_test_batch,
    make_torch_dataloader,
)
from metric_torch import bce_dice_loss, dice_coef
from swin_unetr_mc import build_swin_unetr_mc

SyntaxError: invalid syntax (load_dataset.py, line 1)

# Load hyperparameter

In [ ]:
def _resolve_hparam(value, default):
    if value in (None, "", 0):
        return default
    return value

with open("./hparams.json", "r") as f:
    config = json.load(f)

swin_cfg = config["swin_transformer"]
grid = dict(swin_cfg.get("hparam_grid", {}))
best = dict(swin_cfg.get("best_hparams", {}))
hparams = {**grid, **best}

patch_size = _resolve_hparam(hparams.get("patch_size"), (64, 64, 64))
if isinstance(patch_size, int):
    patch_size = (patch_size,) * 3
else:
    patch_size = tuple(patch_size)

hparams["patch_size"] = patch_size
hparams["batch_size"] = int(_resolve_hparam(hparams.get("batch_size"), 1))
hparams["dropout_rate"] = float(_resolve_hparam(hparams.get("dropout_rate"), 0.2))
hparams["attn_drop_rate"] = float(
    _resolve_hparam(hparams.get("attn_drop_rate"), hparams["dropout_rate"])
 )
hparams["dropout_path_rate"] = float(
    _resolve_hparam(hparams.get("dropout_path_rate"), hparams["dropout_rate"])
 )
hparams["learning_rate"] = float(_resolve_hparam(hparams.get("learning_rate"), 1e-4))
hparams["n_epochs"] = int(_resolve_hparam(hparams.get("n_epochs"), 5))
hparams["model_out"] = str(_resolve_hparam(hparams.get("model_out"), "swin_unetr_mc_best.pth"))
hparams["feature_size"] = int(_resolve_hparam(hparams.get("feature_size"), 24))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("Resolved hparams:", hparams)

Using device: cpu
Resolved hparams: {'patch_size': (64, 64, 64), 'batch_size': 1, 'dropout_rate': 0.2, 'learning_rate': 0.001, 'n_epochs': 5, 'optimizer': 'Adam', 'loss_function': 'bce_dice', 'attn_drop_rate': 0.2, 'dropout_path_rate': 0.2, 'model_out': 'swin_unetr_mc_best.pth', 'feature_size': 24}


# Build datasets

In [ ]:
x_train, y_train = build_dataset(training=True)
x_test, y_test = build_dataset(training=False)

# TODO: Replace this temporary test-as-validation setup with a proper validation split.
train_loader = make_torch_dataloader(x_train, y_train, training=True)
val_loader = make_torch_dataloader(x_test, y_test, training=False)

print("Train samples:", len(x_train), "Val samples:", len(x_test))
print("Train batches:", len(train_loader), "Val batches:", len(val_loader))

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/g/My Drive/CMSC 324 Final project dataset/Training'

# Configure model

## PyTorch Training With External Model Import

In [ ]:
model = build_swin_unetr_mc(
    input_shape=(*hparams["patch_size"], 4),
    out_channels=1,
    feature_size=hparams["feature_size"],
    drop_rate=hparams["dropout_rate"],
    attn_drop_rate=hparams["attn_drop_rate"],
    dropout_path_rate=hparams["dropout_path_rate"],
    force_mc_dropout=True,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=hparams["learning_rate"])

model_out = Path(hparams["model_out"])

best_val_dice = float("-inf")
history = {"train_loss": [], "train_dice": [], "val_loss": [], "val_dice": []}

print(model.__class__.__name__)
print("Checkpoint path:", model_out)

# Train model

In [ ]:
def _prepare_batch(x_batch, y_batch, training):
    if training:
        x_batch, y_batch = augment_torch_batch(x_batch, y_batch)
    else:
        x_batch, y_batch = format_torch_test_batch(x_batch, y_batch)
    return x_batch.to(device), y_batch.to(device)

for epoch in range(1, hparams["n_epochs"] + 1):
    model.train()
    train_losses = []
    train_dices = []

    for xb, yb in tqdm(train_loader, desc=f"Epoch {epoch} train", leave=False):
        xb, yb = _prepare_batch(xb, yb, training=True)
        optimizer.zero_grad(set_to_none=True)
        preds = torch.sigmoid(model(xb))
        loss = bce_dice_loss(yb, preds)
        loss.backward()
        optimizer.step()

        train_losses.append(float(loss.detach().cpu()))
        train_dices.append(float(dice_coef(yb.detach(), preds.detach()).cpu()))

    model.eval()
    val_losses = []
    val_dices = []
    with torch.no_grad():
        for xb, yb in tqdm(val_loader, desc=f"Epoch {epoch} val", leave=False):
            xb, yb = _prepare_batch(xb, yb, training=False)
            preds = torch.sigmoid(model(xb))
            loss = bce_dice_loss(yb, preds)
            val_losses.append(float(loss.cpu()))
            val_dices.append(float(dice_coef(yb, preds).cpu()))

    mean_train_loss = sum(train_losses) / max(len(train_losses), 1)
    mean_train_dice = sum(train_dices) / max(len(train_dices), 1)
    mean_val_loss = sum(val_losses) / max(len(val_losses), 1)
    mean_val_dice = sum(val_dices) / max(len(val_dices), 1)

    history["train_loss"].append(mean_train_loss)
    history["train_dice"].append(mean_train_dice)
    history["val_loss"].append(mean_val_loss)
    history["val_dice"].append(mean_val_dice)

    if mean_val_dice > best_val_dice:
        best_val_dice = mean_val_dice
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "hparams": hparams,
                "best_val_dice": best_val_dice,
            },
            model_out,
        )

    print(
        f"Epoch {epoch}/{hparams['n_epochs']} | "
        f"train_loss={mean_train_loss:.4f} train_dice={mean_train_dice:.4f} | "
        f"val_loss={mean_val_loss:.4f} val_dice={mean_val_dice:.4f}"
    )

# Check the result

In [ ]:
print(f"Best validation Dice: {best_val_dice:.4f}")
print(f"Saved checkpoint: {model_out}")

checkpoint = torch.load(model_out, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

with torch.no_grad():
    sample_x, sample_y = next(iter(val_loader))
    sample_x, sample_y = format_torch_test_batch(sample_x, sample_y)
    sample_x = sample_x.to(device)

    mc_preds = [torch.sigmoid(model(sample_x[:1])) for _ in range(3)]
    mc_stack = torch.stack(mc_preds, dim=0)
    mc_variance = float(mc_stack.var(dim=0).mean().cpu())

print(f"MC dropout variance mean (single sample): {mc_variance:.6f}")